# LifePilot AI — Phase 7: Production Embedding Engine Benchmark

This notebook demonstrates how to load, benchmark, and compare the 4 production-quality embedding models supported by the LifePilot AI Embedding Engine on a Google Colab T4 GPU environment.

### Supported Models:
1. `BAAI/bge-small-en-v1.5` (Dimension: 384)
2. `BAAI/bge-base-en-v1.5` (Dimension: 768)
3. `intfloat/e5-base-v2` (Dimension: 768)
4. `sentence-transformers/all-MiniLM-L6-v2` (Dimension: 384)

## 1. Install Dependencies

In [ ]:
!pip install sentence-transformers torch pandas tabulate

## 2. Benchmark Script Setup

In [ ]:
import time
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

MODELS = [
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
    "sentence-transformers/all-MiniLM-L6-v2"
]

sample_texts = [
    "LifePilot AI remembers users across conversations.",
    "Retrieval-Augmented Generation processes documents like PDF and Word files.",
    "Vector databases flat indexes perform hybrid metadata filtering similarity calculations.",
    "Enterprise modular clean code requires high test coverage and observability metrics.",
    "Production embeddings engine caches computed vectors using secure content-hashing."
] * 10 # 50 texts for batch throughput measurement

## 3. Run Benchmark Suite

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

results = []

for model_name in MODELS:
    print(f"\nEvaluating: {model_name}...")

    # Clear CUDA cache before load
    if device == "cuda":
        torch.cuda.empty_cache()
        vram_before = torch.cuda.memory_allocated() / (1024 * 1024)
    else:
        vram_before = 0

    # Measure load time
    start_load = time.perf_counter()
    model = SentenceTransformer(model_name, device=device)
    load_time_ms = (time.perf_counter() - start_load) * 1000

    # Measure memory usage
    if device == "cuda":
        vram_after = torch.cuda.memory_allocated() / (1024 * 1024)
        vram_usage_mb = vram_after - vram_before
    else:
        vram_usage_mb = 0

    # Single encoding latency
    latencies = []
    for text in sample_texts[:5]:
        t0 = time.perf_counter()
        model.encode(text, convert_to_numpy=True)
        latencies.append((time.perf_counter() - t0) * 1000)
    avg_single_latency_ms = sum(latencies) / len(latencies)

    # Batch throughput (all 50 texts)
    t0 = time.perf_counter()
    embeddings = model.encode(sample_texts, batch_size=16, convert_to_numpy=True)
    batch_time_ms = (time.perf_counter() - t0) * 1000
    throughput_sec = len(sample_texts) / (batch_time_ms / 1000)

    results.append({
        "Model Name": model_name,
        "Dimension": model.get_sentence_embedding_dimension(),
        "Model Load Time (ms)": round(load_time_ms, 2),
        "GPU memory (MB)": round(vram_usage_mb, 2),
        "Avg Single Latency (ms)": round(avg_single_latency_ms, 2),
        "Batch Throughput (texts/sec)": round(throughput_sec, 2)
    })

df = pd.DataFrame(results)
df

## 4. Export Benchmarking Metrics

In [ ]:
df.to_csv("embedding_benchmark_results.csv", index=False)
print("Exported results to embedding_benchmark_results.csv")